# 📝 LangChain 에이전트와 도구 과제 LV2 정답 — RAG 체인·SQL·외부 API (강사용)

각 문제의 **모범답안 + 해설**입니다. 경로는 `../../day19_LangChain_에이전트_툴/data/` 입니다.

- 학생이 **만든 도구**·로컬 검색은 직접 호출로 채점합니다(모델 미개입 — 결정적).
- **에이전트 궤적**·**구조화 출력**은 실제 모델을 부르므로 **타입·구조**로만 채점합니다 — 도구 호출 횟수·감성 판정·답 문장은 실행마다 달라집니다.
- SQL 문제는 **준비물이 없습니다** — SQL 단원에서 배운 sqlite 를 쓰므로 접속 정보가 필요 없습니다.

아래 준비 셀들을 먼저 실행하세요(임베딩 모델 로딩에 잠시 걸립니다).

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../../day19_LangChain_에이전트_툴/.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 서점 FAQ 문서를 LangChain 부품으로 색인합니다(임베딩 모델을 내려받느라 처음 한 번은 잠시 걸립니다).
import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

faq_df = pd.read_csv('../../day19_LangChain_에이전트_툴/data/bookstore_faq.csv')

# 표의 한 행 = Document 하나. page_content 는 검색 대상 본문, metadata 는 함께 붙일 꼬리표입니다.
faq_docs = [Document(page_content=row.text, metadata={'id': row.id, 'title': row.title})
            for row in faq_df.itertuples()]

embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')   # 임베딩 단원에서 쓴 그 한국어 문장 임베딩 모델

# ids 를 함께 넘기면 같은 id 는 덮어쓰기가 됩니다 -> 이 셀을 여러 번 실행해도 문서가 중복되지 않습니다.
faq_store = Chroma.from_documents(faq_docs, embeddings, collection_name='bookstore_faq',
                                  ids=faq_df['id'].tolist())
faq_retriever = faq_store.as_retriever(search_kwargs={'k': 2})   # 질문마다 가장 가까운 2개를 돌려주는 검색기

print('색인 완료 — 문서 수:', len(faq_docs))

In [ ]:
# [제공 코드] 에이전트 공통 준비
from langchain.agents import create_agent
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool          # 3번에서 도구를 직접 만들 때 씁니다

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## 1. RAG 체인 직접 조립하기
**배경**: 교안에서 본 것처럼, 도구도 에이전트도 없이 **검색 → 프롬프트 → 모델 → 문자열**을 한 줄로 잇는 **체인**을 직접 조립합니다. 맨 위 준비 셀이 만들어 둔 **`faq_retriever`** 를 씁니다.

**요구사항**: 두 가지를 만드세요.

**(1) 기본 RAG 체인 `rag_chain`**

- `ChatPromptTemplate` 로 프롬프트를 만드세요. **`{context}`** 와 **`{question}`** 두 자리를 두고, *주어진 자료에 있는 내용만으로 한국어로 답하라*는 지시를 담습니다.
- 체인은 **검색 결과를 글로 합친 것**을 `context` 에, **질문 그대로**를 `question` 에 채운 뒤 프롬프트 → 모델 → 출력파서를 차례로 잇습니다. 질문을 그대로 흘려보내는 데에는 **`RunnablePassthrough`**, 최종 결과를 문자열로 받는 데에는 **`StrOutputParser`** 를 씁니다.
- 검색 결과를 한 덩어리 글로 합치는 **`format_docs`** 는 아래 제공 셀에 있습니다.
- 질문 **"전자책은 몇 대의 기기에서 볼 수 있나요?"** 로 `invoke` 한 결과를 변수 **`answer1`** 에 담으세요(문자열입니다).

**(2) 근거를 함께 돌려주는 체인 `chain_with_sources`**

- 실무에서 **출처 없는 RAG 답**은 쓰기 어렵습니다. **`RunnableParallel`** 로 두 갈래를 묶어 답과 근거를 함께 받으세요 — **`'answer'`** 에는 (1)의 체인을, **`'sources'`** 에는 검색기를 그대로 둡니다.
- 같은 질문 **"전자책은 몇 대의 기기에서 볼 수 있나요?"** 로 `invoke` 한 결과를 변수 **`result1`** 에 담으세요. `result1['answer']` 는 문자열, `result1['sources']` 는 **`Document` 목록**이 됩니다.

**예시**: `answer1` 은 전자책 기기 수를 설명하는 한국어 문장입니다(**문장은 실행할 때마다 달라지므로** 채점은 **타입·구조**로만 합니다). `result1['sources']` 의 각 `Document` 에는 `metadata['title']` 이 붙어 있어 어느 FAQ 에서 왔는지 알 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 딕셔너리로 두 자리를 채우고, 파이프로 프롬프트·모델·파서를 잇는다.

세부구현:
1. ChatPromptTemplate 로 자료와 질문 자리를 가진 프롬프트를 만든다.
   1-1. 자료에 있는 내용만으로 답하라는 지시를 넣는다.
2. context 자리에는 검색기와 format_docs 를 이어 붙인 것을, question 자리에는
   입력을 그대로 흘려보내는 부품을 둔다.
3. 그 딕셔너리에 프롬프트·모델·문자열 출력파서를 차례로 이어 체인을 만든다.
4. 답과 근거를 함께 받는 체인은 두 갈래를 나란히 묶는 부품으로 만든다.
   4-1. 한 갈래는 앞에서 만든 체인, 다른 갈래는 검색기 그대로다.
5. 두 체인에 같은 질문 문자열을 넣어 결과를 각각 담는다.
```

</details>

In [ ]:
# [제공 코드] 검색 결과(Document 목록)를 프롬프트에 넣을 한 덩어리 글로 합칩니다 — 실행만 하세요.
def format_docs(docs):
    """검색된 Document 들을 제목과 함께 한 덩어리 글로 합친다."""
    return '\n\n'.join(f"[{d.metadata['title']}] {d.page_content}" for d in docs)


print('준비 완료 —', format_docs(faq_retriever.invoke('전자책'))[:40], '...')

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# '자료에 있는 내용만으로' 를 못 박아야 모델이 아는 척으로 지어내지 않는다
prompt1 = ChatPromptTemplate.from_template(
    '너는 온라인 서점 상담원이다. 아래 자료에 있는 내용만으로 한국어로 답하라.\n\n'
    '자료:\n{context}\n\n질문: {question}')

# 딕셔너리의 각 갈래에 같은 입력(질문 문자열)이 들어간다
#  context 갈래: 검색기 -> 합치기 / question 갈래: 입력을 그대로 통과
rag_chain = ({'context': faq_retriever | format_docs, 'question': RunnablePassthrough()}
             | prompt1 | model | StrOutputParser())
answer1 = rag_chain.invoke('전자책은 몇 대의 기기에서 볼 수 있나요?')
print(answer1)

# 답과 근거를 함께 — sources 갈래는 검색기를 그대로 둬 Document 목록이 그대로 온다
chain_with_sources = RunnableParallel(answer=rag_chain, sources=faq_retriever)
result1 = chain_with_sources.invoke('전자책은 몇 대의 기기에서 볼 수 있나요?')
print('근거:', [d.metadata['title'] for d in result1['sources']])

In [ ]:
# [자가채점]
# 실호출이라 문장은 매번 다르다 — 타입과 구조만 본다
assert isinstance(answer1, str) and len(answer1.strip()) > 0
assert isinstance(result1, dict) and set(result1) == {'answer', 'sources'}, \
    "두 갈래의 이름은 'answer' 와 'sources' 입니다"
assert isinstance(result1['answer'], str) and len(result1['answer'].strip()) > 0
assert isinstance(result1['sources'], list) and len(result1['sources']) >= 1
assert all(isinstance(d, Document) for d in result1['sources']), 'sources 는 Document 목록입니다'
assert all('title' in d.metadata for d in result1['sources'])
# sources 가 정말 검색기에서 왔는지 — 같은 질문을 검색기에 직접 넣어 대조한다(검색은 결정적이다)
want1 = [d.metadata['id'] for d in faq_retriever.invoke('전자책은 몇 대의 기기에서 볼 수 있나요?')]
assert [d.metadata['id'] for d in result1['sources']] == want1, \
    'sources 갈래에 검색기를 그대로 두세요'
print('✅ 통과!')

**해설**: 체인은 **경로가 고정**입니다 — 무엇을 묻든 검색이 먼저 돌고, 그 결과가 프롬프트에 박혀 모델로 갑니다. 인사말을 넣어도 검색이 돕니다 — 교안 2절 끝에서 본 그 성질입니다. 같은 RAG 라도 **누가 결정하느냐**가 다릅니다. 자료 질문만 들어오는 창구라면 체인이 단순하고 빠르며, 잡담과 자료 질문이 섞여 들어오면 도구 방식이 낫습니다.

**딕셔너리가 왜 체인의 일부가 되나**: `{'context': ..., 'question': ...}` 는 두 갈래를 **나란히 실행**하는 부품이 됩니다. 같은 입력(질문 문자열)이 양쪽에 들어가, 한쪽은 검색을 거쳐 자료가 되고 다른 쪽은 그대로 질문으로 남습니다. `RunnablePassthrough` 가 그 '그대로 통과' 를 맡습니다.

**근거를 함께 돌려주는 이유**: 답만 있으면 사용자는 **그 말이 맞는지 확인할 길이 없습니다.** `RunnableParallel` 로 검색기를 한 갈래에 그대로 두면 답과 근거 문서를 한 번에 받습니다. 실무 RAG 화면의 '출처 보기' 가 이렇게 만들어집니다.

**채점 기준**: 모델 문장은 매번 다르므로 **타입·구조**만 봅니다. 다만 `sources` 는 검색 결과라 **결정적**이어서, 채점이 같은 질문을 검색기에 직접 넣어 **id 목록까지 대조**합니다.

## 2. Text-to-SQL — 주문 건수 조회, 그리고 답을 스키마로
**배경**: 자연어 질문을 SQL 로 바꿔 답하게 합니다. **SELECT 전용** 도구와 **스키마 안내 프롬프트**를 함께 씁니다. 그다음, 같은 질문의 답을 **문장이 아니라 데이터**로 받아 봅니다.

**요구사항**: 두 가지를 만드세요.

**(1) 궤적 읽기**

- 아래 준비 셀들(`run_select` 도구·`SCHEMA_PROMPT`·데이터베이스·`OrderAnswer` 스키마)을 먼저 실행하세요.
- `create_agent(model, [run_select], system_prompt=SCHEMA_PROMPT)` 로 에이전트를 만들고, 질문 **"c1 고객이 주문한 건수는 모두 몇 건인가요?"** 로 `invoke` 한 결과를 변수 **`res2`** 에 담으세요.
- 궤적에서 **ToolMessage** 만 골라 변수 **`tool_msgs2`** 에 담으세요.

**(2) 도구도 쓰고, 답도 스키마로 — `response_format`**

- (1)의 최종 답은 **자유 문장**이라 그대로는 표에 넣지 못합니다. 교안 01 3절에서 본 것처럼 `create_agent` 에 **`response_format=ProviderStrategy(OrderAnswer, strict=True)`** 를 더해 에이전트를 하나 더 만드세요(임포트는 `from langchain.agents.structured_output import ProviderStrategy`).
- **같은 질문** "c1 고객이 주문한 건수는 모두 몇 건인가요?" 로 `invoke` 한 결과를 변수 **`res2b`** 에 담고, 거기서 정형 결과를 꺼내 변수 **`report2`** 에 담으세요. 정형 결과는 결과 딕셔너리의 **`'structured_response'`** 열쇠에 들어 있습니다.

**예시**: c1 고객의 주문은 3건이라 (1)의 도구 결과 어딘가에 **'3'** 이 들어 있습니다(모델이 표를 먼저 둘러보느라 조회를 여러 번 할 수도 있습니다). (2)의 `report2` 는 `OrderAnswer` 객체이고, `report2.order_count` 는 **3**, `report2.sql` 에는 실행한 **`select`** 문이, `report2.answer` 에는 사용자에게 보여 줄 한국어 한 문장이 들어 있습니다. 문장 내용은 실행마다 다릅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 스키마 프롬프트를 준 SQL 에이전트로 질문을 실행하고, 도구 결과를 읽는다.
- 답까지 정형화하려면 에이전트를 만들 때 인자를 하나 더 준다 - 도구 목록은 그대로다.

세부구현:
1. create_agent 에 [run_select] 와 system_prompt=SCHEMA_PROMPT 를 준다.
2. 질문을 그대로 invoke 해 res2 에 담는다.
3. ToolMessage 만 골라 tool_msgs2 에 담는다.
4. 같은 인자에 response_format 을 더해 두 번째 에이전트를 만들고 같은 질문을 invoke 한다.
   4-1. 결과 딕셔너리에 키가 하나 더 생겨 있다 - 궤적은 그대로 남는다.
5. 그 키에서 정형 결과를 꺼내 report2 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 데이터베이스 준비 — SQL 단원에서 배운 sqlite 를 그대로 씁니다(접속 정보가 필요 없습니다).
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('output') / 'bookstore.db'
DB_PATH.parent.mkdir(exist_ok=True)
DB_PATH.unlink(missing_ok=True)          # 여러 번 실행해도 늘 같은 초기 상태에서 시작합니다

_conn = sqlite3.connect(DB_PATH, isolation_level=None)   # isolation_level=None : 실행 즉시 저장
_conn.execute('pragma foreign_keys = on')                # 외래키 검사를 켭니다(기본값은 꺼짐)
_conn.executescript(Path('../../day19_LangChain_에이전트_툴/data/setup_day19.sql').read_text(encoding='utf-8'))
_conn.execute('pragma foreign_keys = on')                # executescript 뒤에 한 번 더 켭니다

# 에이전트에게 줄 연결은 따로 만들고 '읽기 전용'으로 엽니다 — 모델이 무슨 SQL 을 만들든 쓰기가 막힙니다.
#  check_same_thread=False : 에이전트는 도구를 별도 스레드에서 실행하므로 이 옵션이 없으면 도구가 전부 실패합니다.
_ro_conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True,
                           isolation_level=None, check_same_thread=False)


def run_query(sql):
    """SELECT 결과를 DataFrame 으로 돌려준다(사람이 눈으로 확인할 때 쓴다)."""
    cur = _conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print('데이터베이스 준비 완료 —', DB_PATH)

In [ ]:
# [제공 코드] 데이터베이스 조회 도구 — 에이전트가 이 도구로 SQL 을 실행합니다.
#  가드가 두 겹입니다: (1) 여기서 문장을 검사하고 (2) 연결 자체가 읽기 전용입니다.
from langchain_core.tools import tool


@tool
def run_select(sql: str) -> str:
    """읽기 전용 SQL(SELECT) 한 문장을 실행하고 결과를 문자열로 돌려준다. SELECT 한 문장이 아니면 거부한다."""
    stmt = sql.strip().rstrip(';')          # 끝의 세미콜론 하나는 흔한 표기라 허용한다
    # 세미콜론이 남아 있으면 문장이 둘 이상이라는 뜻 — 'select 1; delete ...' 를 막는다.
    if not stmt.lower().startswith('select') or ';' in stmt:
        return '거부: 이 도구는 SELECT 조회 한 문장만 실행할 수 있습니다.'
    try:
        return str(_ro_conn.execute(stmt).fetchall())   # 검사한 문장을 그대로 실행한다
    except Exception as e:
        return f'에러: {e}'                             # 에러도 문자열로 — 모델이 읽고 고쳐 다시 시도한다


print('SQL 도구 준비:', run_select.name)

In [ ]:
# [제공 코드] Text-to-SQL 용 스키마 안내 프롬프트
SCHEMA_PROMPT = (
    '너는 온라인 서점 데이터베이스 조회를 돕는 도우미다. run_select 도구로 SELECT 문만 실행해 답하라. 표 스키마는 다음과 같다. bs_customer(customer_id, name, grade, city): 고객. bs_book(book_id, title, author, genre, price, stock): 도서. bs_order(order_id, customer_id, book_id, quantity, order_date): 주문. 조회 결과를 바탕으로 한국어로 간단히 답하라.'
)

In [ ]:
# [제공 코드] 2번 (2)에서 쓸 결과 스키마 — 이 셀은 실행만 하세요.
from pydantic import BaseModel, Field


class OrderAnswer(BaseModel):
    """주문 건수 문의 한 건을 처리한 결과."""

    sql: str = Field(description="조회에 사용한 SELECT 문")
    # 이 칸은 모델이 지어내는 값이 아니라 '도구가 알려 준 값'이다.
    order_count: int = Field(description="도구가 알려 준 주문 건수")
    answer: str = Field(description="사용자에게 보여 줄 한국어 한 문장")


print("스키마 준비 완료")

In [ ]:
# 표 구조를 모르면 모델이 없는 열을 지어낸다 — 스키마 안내는 system_prompt 로 준다
agent2 = create_agent(model, [run_select], system_prompt=SCHEMA_PROMPT)
res2 = agent2.invoke({'messages': [{'role': 'user', 'content': 'c1 고객이 주문한 건수는 모두 몇 건인가요?'}]})
tool_msgs2 = [m for m in res2['messages'] if isinstance(m, ToolMessage)]
print('SQL 결과:', [m.content for m in tool_msgs2])
print('최종 답:', res2['messages'][-1].text)

# (2) 도구 목록도 프롬프트도 그대로 — 인자 하나(response_format)만 더 준다
from langchain.agents.structured_output import ProviderStrategy

report_agent2 = create_agent(model, [run_select], system_prompt=SCHEMA_PROMPT,
                             response_format=ProviderStrategy(OrderAnswer, strict=True))
res2b = report_agent2.invoke({'messages': [{'role': 'user', 'content': 'c1 고객이 주문한 건수는 모두 몇 건인가요?'}]})
# 궤적은 그대로 남고, 결과 딕셔너리에 키가 하나 더 생긴다
print('결과의 키:', list(res2b))
report2 = res2b['structured_response']
print(report2)

In [ ]:
# [자가채점]
assert tool_msgs2 == [m for m in res2['messages'] if isinstance(m, ToolMessage)]
assert res2['messages'][0].text.strip() == 'c1 고객이 주문한 건수는 모두 몇 건인가요?', '지문의 질문을 그대로 넣어 실행하세요'
assert len(tool_msgs2) >= 1, 'SQL 도구가 한 번도 불리지 않았습니다'
assert any('3' in str(m.content) for m in tool_msgs2)   # c1 의 주문 건수 3
assert isinstance(res2['messages'][-1].text, str)

# (2) 정형 결과 — 값이 아니라 '모양'을 스키마가 보장한다
assert report2 is res2b['structured_response'], 'report2 는 res2b 에서 꺼낸 정형 결과여야 합니다'
assert isinstance(report2, OrderAnswer)
# response_format 을 줘도 도구는 그대로 불린다 — 궤적이 남아 있는지 본다
assert any(isinstance(m, ToolMessage) for m in res2b['messages']), \
    'response_format 을 줘도 도구 궤적은 남습니다 — 도구가 불리지 않았습니다'
assert report2.order_count == 3, 'order_count 는 도구가 알려 준 주문 건수(3)여야 합니다'
assert 'select' in report2.sql.lower(), 'sql 칸에는 조회에 쓴 SELECT 문이 들어갑니다'
assert isinstance(report2.answer, str) and report2.answer.strip()
print('✅ 통과!')

**해설**: 모델이 `select count(*) from bs_order where customer_id='c1'` 같은 SQL 을 스스로 만들어 실행합니다. 우리는 **스키마만** 알려 줬습니다. 결과 값(3)이 도구 결과에 담깁니다.

**(2) 가 메우는 반쪽**: `with_structured_output` 은 답을 칸에 담아 주지만 **도구를 못 쓰고**, 에이전트는 도구를 쓰지만 답이 **자유 문장**입니다. `response_format` 은 둘을 합칩니다 — 도구를 다 쓴 **뒤** 마지막 답만 스키마에 담으므로, `res2b['messages']` 의 궤적은 그대로 남고 `res2b['structured_response']` 라는 **키가 하나 더** 생깁니다. `report2.order_count` 는 모델이 지어낸 수가 아니라 **도구가 알려 준 값**이라, 이 한 칸을 뽑아 바로 대시보드에 넣을 수 있습니다.

**채점 기준**: 모델이 쓰는 SQL 문장은 실행마다 다를 수 있고, 표를 먼저 둘러보느라 조회를 여러 번 할 수도 있습니다. 그래서 **첫 번째** 도구 결과가 아니라 **도구 결과들 중 어딘가에** 정답 값이 있는지로 채점합니다. (2)도 문장은 보지 않고 **타입·구조**를 봅니다 — 다만 `order_count` 만은 도구가 돌려준 값이라 **3** 으로 못박습니다.

## 3. 외부 API 를 도구로 — 오늘의 엔화 환율
**배경**: 지금까지 만든 도구는 전부 **노트북 안의 값**으로 답했습니다. 오늘의 값이 필요하면 **인터넷에 직접 물어봐야** 합니다. 이번엔 달러가 아니라 **엔화**로, 그리고 **금액을 인자로 받는** 도구를 직접 만듭니다.

`https://api.frankfurter.dev/v1/latest` 에 `amount`·`from`·`to` 를 붙여 요청하면 환산 결과가 옵니다 (키가 필요 없는 공개 API). 예를 들어 `{'amount': 100, 'from': 'JPY', 'to': 'KRW'}` 로 요청하면 응답 JSON 의 `rates['KRW']` 에 **100엔에 해당하는 원화**가 들어 있습니다.

> **인터넷은 언제든 끊깁니다.** 교안 01 4절의 **도구 설계 원칙 3 — 실패는 예외가 아니라 문자열로 돌려준다** 를 이 도구에 적용하세요. 예외를 밖으로 던지면 에이전트가 그 자리에서 멈추지만, 문자열이면 모델이 그것을 읽고 사용자에게 안내하거나 다시 시도할 수 있습니다.

**요구사항**:
- `@tool` 을 붙인 함수 **`jpy_krw_rate(amount_jpy: int) -> str`** 를 정의하세요. docstring 을 한국어로 적습니다.
- `import requests` 한 뒤 함수 안에서 **`requests.get(...)`** 으로 위 주소에 요청하세요. `params={'amount': amount_jpy, 'from': 'JPY', 'to': 'KRW'}` 와 **`timeout=10`** 을 줍니다.
- 요청과 `raise_for_status()`·`.json()` 을 **`try` 안에** 두고, **`except requests.RequestException`** 으로 실패를 잡으세요. 잡았을 때는 **예외를 다시 던지지 말고**, 왜 실패했는지 담은 **문자열**을 돌려줍니다.
- 성공했을 때 반환 문자열은 **`f'{amount_jpy}엔 = {round(원화)}원'`** 형식입니다(원화는 `rates['KRW']`, **반올림해 정수**로).
- `create_agent(model, [jpy_krw_rate])` 로 에이전트를 만들고, 질문 **"300엔은 우리 돈으로 얼마야?"** 로 `invoke` 한 결과를 변수 **`res3`** 에 담은 뒤, 불린 도구 이름 목록을 **`used3`** 에 담으세요.

**예시**: `jpy_krw_rate.invoke({'amount_jpy': 100})` → `'100엔 = 921원'` 같은 문자열(**환율은 날마다 바뀌므로 숫자는 오늘 값**입니다). 요청이 실패하면 `'환율을 가져오지 못했습니다: ...'` 처럼 **사정을 알리는 문자열**이 나옵니다. `used3` 에는 `'jpy_krw_rate'` 가 들어 있습니다.

> **채점 안내**: 자가채점은 성공 경로뿐 아니라 **실패 경로도** 확인합니다 — `requests.get` 이 실패하도록 잠깐 바꿔 놓고 도구를 불러, 예외가 밖으로 나오지 않고 **문자열**이 돌아오는지 봅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 데이터 수집 단원에서 배운 requests 요청과 JSON 읽기를 @tool 함수 안에 넣되, 끊길 때에 대비해 try 로 감싼다.

세부구현:
1. requests 를 임포트하고 @tool 과 타입힌트·docstring 을 단다.
2. try 안에서 params 에 amount·from·to 를 넣고 timeout 을 준 뒤,
   raise_for_status 로 실패를 확인하고 응답 JSON 을 읽는다.
3. except 로 requests 의 요청 계열 예외를 잡고, 다시 던지지 말고
   실패를 알리는 문자열을 반환한다(도구 설계 원칙 3).
4. 성공하면 응답에서 원화 값을 꺼내 반올림해 지정된 형식의 문자열로 만든다.
5. 만든 도구 하나로 에이전트를 만들고 질문을 invoke 한 뒤 ToolMessage 의 name 을 모은다.
```

</details>

In [ ]:
# 데이터 수집 단원에서 배운 requests 를 그대로 쓴다 — 달라지는 것은 이 호출이 @tool 안에 있다는 것뿐
import requests


@tool
def jpy_krw_rate(amount_jpy: int) -> str:
    """엔화 금액(amount_jpy)을 오늘의 환율로 원화로 환산해 돌려준다."""
    try:
        res = requests.get('https://api.frankfurter.dev/v1/latest',
                           params={'amount': amount_jpy, 'from': 'JPY', 'to': 'KRW'},
                           timeout=10)      # 상대 서버가 멈춰도 노트북은 안 멈추게
        res.raise_for_status()              # 4xx·5xx 면 예외를 낸다(조용한 실패를 만들지 않는다)
        krw = res.json()['rates']['KRW']    # 응답 JSON 에서 필요한 값 하나만 꺼낸다
    except requests.RequestException as e:
        # 도구 설계 원칙 3 — 실패도 '문자열'로 돌려준다.
        #   예외를 밖으로 던지면 에이전트가 그 자리에서 멈춘다. 문자열이면 모델이 그것을 읽고
        #   사용자에게 안내하거나 다시 시도한다. 인터넷은 언제든 끊길 수 있다.
        return f'환율을 가져오지 못했습니다: {e}. 잠시 뒤 다시 시도해 주세요.'
    return f'{amount_jpy}엔 = {round(krw)}원'  # 모델이 읽을 만큼만 짧게 돌려준다


print(jpy_krw_rate.invoke({'amount_jpy': 100}))

agent5 = create_agent(model, [jpy_krw_rate])
res3 = agent5.invoke({'messages': [{'role': 'user', 'content': '300엔은 우리 돈으로 얼마야?'}]})
used3 = [m.name for m in res3['messages'] if isinstance(m, ToolMessage)]
print(used3)
print('최종 답:', res3['messages'][-1].text)

In [ ]:
# [자가채점]
import requests as rq

# 채점도 같은 API 에 직접 물어본다 — 값을 손으로 박아 두면 오늘 환율과 어긋나 걸린다
#  채점 자신이 망을 타므로, 여기서 끊기면 '내 코드가 틀린 건가' 하고 헤매게 된다.
#  그래서 채점의 요청도 감싸서, 망 문제일 때는 그렇다고 말해 준다.
try:
    ref3 = rq.get('https://api.frankfurter.dev/v1/latest',
                  params={'amount': 100, 'from': 'JPY', 'to': 'KRW'}, timeout=10).json()
except Exception as e:
    raise AssertionError(
        f'채점이 환율 API 에 닿지 못했습니다({type(e).__name__}) — 인터넷 연결을 확인하세요. '
        '여러분의 코드 문제가 아닙니다.') from None

got3 = jpy_krw_rate.invoke({'amount_jpy': 100})
assert isinstance(got3, str)
assert '100엔' in got3
assert f"{round(ref3['rates']['KRW'])}원" in got3, '오늘 환율로 환산한 값이 보이지 않습니다'

# 실패 경로도 본다 — 요청이 실패하도록 requests.get 을 잠깐 바꿔 놓고 도구를 부른다
saved_get = rq.get


def failing_get(*args, **kwargs):
    """채점용 모의 상황 — 인터넷이 끊긴 것처럼 요청을 실패시킨다."""
    raise rq.RequestException('연결 실패(채점용 모의 상황)')


rq.get = failing_get
try:
    fail3 = jpy_krw_rate.invoke({'amount_jpy': 100})
finally:
    rq.get = saved_get                     # 무슨 일이 있어도 원래대로 되돌린다
assert isinstance(fail3, str), '실패해도 예외를 던지지 말고 문자열을 돌려주세요(도구 설계 원칙 3)'

assert used3 == [m.name for m in res3['messages'] if isinstance(m, ToolMessage)]
assert res3['messages'][0].text.strip() == '300엔은 우리 돈으로 얼마야?', '지문의 질문을 그대로 넣어 실행하세요'
assert 'jpy_krw_rate' in used3, '에이전트가 도구를 부르지 않았습니다'
print('✅ 통과!')

**해설**: 도구 안에서 무엇을 하든 모델에게는 똑같은 도구입니다 — 계산이든, DB 조회든, **외부 API 호출**이든. 새로 배운 문법은 없고, 데이터 수집 단원의 `requests` 를 `@tool` 안으로 옮겼을 뿐입니다.

**세 가지 습관**: `timeout` 을 주어 상대 서버가 멈춰도 노트북이 멈추지 않게 하고, 응답 JSON 전체가 아니라 **필요한 값만 짧은 문장으로** 돌려주며(도구 결과는 그대로 모델에게 다시 들어가 비용이 됩니다), **실패를 문자열로** 돌려줍니다.

**왜 `try` 로 감쌌나 — 도구 설계 원칙 3**: `raise_for_status()` 를 그냥 두면 서버가 5xx 를 낼 때 예외가 도구 밖으로 튀어나가고, **에이전트는 그 자리에서 멈춥니다**(사용자는 답 대신 스택 트레이스를 봅니다). 예외를 잡아 `'환율을 가져오지 못했습니다: ...'` 같은 문자열로 돌려주면 그 문장이 **ToolMessage 로 모델에게 되돌아가고**, 모델은 사정을 설명하거나 다시 시도합니다. `raise_for_status()` 를 빼라는 뜻이 아닙니다 — **던지되 그 자리에서 받아 문자열로 바꾸는** 것입니다. 인터넷을 쓰는 도구에서는 실패가 예외 상황이 아니라 **평범한 경우의 수**입니다.

**채점 기준**: 환율은 날마다 바뀌므로 값을 못박을 수 없습니다. 그래서 **채점 셀이 같은 API 에 직접 물어보고** 그 값이 여러분의 도구 결과 안에 있는지 확인합니다 — 숫자를 손으로 적어 두면 다음 날 바로 어긋납니다. 실무의 회귀 테스트도 이런 식으로 **기준값을 그때그때 구해** 비교합니다.

이어서 **실패 경로**도 채점합니다. `requests.get` 을 잠깐 실패하는 함수로 바꿔 놓고 도구를 불러, 예외가 밖으로 나오지 않고 문자열이 오는지 봅니다(끝나면 `finally` 로 원래대로 되돌립니다). 실무에서도 **끊긴 상황을 일부러 만들어** 시험합니다 — 그 경로는 평소에 실행되지 않아 조용히 썩기 때문입니다.

## 4. 조각으로 잘린 안내문을 넓은 문맥으로 답하기
**배경**: 준비 셀의 `faq_retriever` 는 FAQ **한 편을 통째로** 담고 있습니다. 실무 문서는 그렇게 짧지 않아 **조각으로 잘라** 색인하는데, 그러면 검색이 집어 온 조각 하나가 **문장 중간에서 끊겨** 답을 만들 재료로 모자랍니다. 교안 1·2절에서 본 것처럼 **조각에 번호를 달아 두었다가 앞뒤를 되찾아** 자료를 넓힙니다.

**요구사항**: 두 가지를 만드세요. 준비 셀이 만들어 둔 **`faq_docs`**(서점 FAQ 22편)와 **`embeddings`** 를 그대로 씁니다.

**(1) 조각 색인과 이웃 함수**

- `RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10, add_start_index=True)` 로 **문서를 하나씩** 잘라, 조각마다 `metadata` 에 **`chunk_no`**(그 문서에서 몇 번째인지, 0부터)와 **`chunk_total`**(그 문서의 조각 수)을 넣고 전부 **`chunks4`** 에 모으세요.
- 조각 id 는 **`'문서id-조각번호'`** 형태로 만들어 `Chroma.from_documents(..., collection_name='bookstore_chunk_lv2', ids=...)` 로 담고, 그 저장소를 **`store4`** 에, `as_retriever(search_kwargs={'k': 1})` 로 만든 검색기를 **`retriever4`** 에 담으세요.
- 검색된 조각의 **앞뒤 `window` 개**를 같은 문서에서 꺼내 **조각 번호 순서로** 돌려주는 함수 **`neighbors4(hit, window=1)`** 를 만드세요. 돌려줄 것은 **`(metadata, 본문)` 쌍의 목록**입니다.
  - 꼬리표로 고를 때는 검색(`invoke`)이 아니라 **`store4.get(where=...)`** 를 씁니다. 조건이 둘이면 **`{'$and': [...]}`** 로 묶고, 목록 안에 있는지는 **`{'$in': [...]}`** 입니다.
  - 문서의 처음·끝을 넘는 번호는 **빼야** 합니다(`chunk_total` 이 그래서 필요합니다).
  - **`get` 은 순서를 보장하지 않습니다** — `chunk_no` 로 정렬해야 원문 순서가 됩니다.

**(2) 넓은 문맥 체인**

- **`search_wide4(question)`** — `retriever4` 로 검색하고, 집힌 조각마다 `neighbors4` 로 앞뒤를 붙여 **`{'context': ..., 'question': ..., 'sources': ...}`** 를 돌려줍니다(`sources` 는 검색이 집은 `Document` 목록 그대로).
- **`answer_wide4(filled)`** — 제공된 **`wide_prompt`** 에 모델과 `StrOutputParser` 를 이어 답을 만들고 **`{'answer': ..., 'sources': ...}`** 를 돌려줍니다.
- 두 함수를 **`RunnableLambda`** 로 감싸 `|` 로 이어 **`wide_rag4`** 를 만드세요.
- 질문 **"매장 수령은 준비된 뒤 며칠 안에 가야 하나요?"** 로 (a) `search_wide4` 를 직접 불러 결과를 **`wide4`** 에, (b) `wide_rag4.invoke(...)` 결과를 **`out4`** 에 담으세요(`wide4` 는 **자료가 어떻게 넓어졌는지 눈으로 보려고** 앞 단계만 따로 받아 두는 것입니다 — 실제 서비스에서는 체인만 부릅니다).

**예시**: 이 질문이 집는 조각은 `'매장 수령은 배송비가 들지 않으며 … 준비 후'` 에서 **끊겨 있어 며칠인지 알 수 없습니다.** 앞뒤를 붙이면 `wide4['context']` 에 `'7일이 지나면 자동으로 주문이 취소됩니다'` 가 들어오고, `out4['answer']` 는 7일을 말하는 한국어 문장이 됩니다(문장은 실행마다 다릅니다).

> **채점 안내**: 모델 문장은 **타입·구조**로만 봅니다. 대신 자료(`wide4['context']`)는 **코드로만 만들어져 결정적**이라, 집힌 조각에는 없던 문장이 자료에 들어왔는지까지 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문서를 하나씩 잘라 번호를 달고, 검색 뒤에는 그 번호로 이웃을 꺼내 이어 붙인다.
- 자료를 넓히는 단계와 답을 만드는 단계를 나눠 파이프로 잇는다.

세부구현:
1. 스플리터를 만들고 문서 목록을 돌며 문서 하나씩 자른다.
   1-1. 그 문서의 조각들을 돌며 순번과 총 개수를 꼬리표에 넣는다.
2. 문서 id 와 조각 번호를 이어 붙여 조각 id 목록을 만들고 저장소에 담는다.
3. 이웃 함수는 원하는 번호 목록을 먼저 만든다 - 범위를 벗어난 번호는 거른다.
   3-1. 저장소에서 꼬리표 조건으로 꺼낸 뒤 조각 번호로 정렬해 돌려준다.
4. 자료를 만드는 함수는 검색 결과를 돌며 이웃을 이어 붙여 한 덩어리 글을 만들고,
   질문과 근거를 같은 딕셔너리에 함께 담는다.
5. 답을 만드는 함수는 제공된 프롬프트에 모델과 문자열 출력파서를 이어 부른다.
6. 두 함수를 각각 감싸 파이프로 잇고, 지문의 질문으로 실행한다.
```

</details>

In [ ]:
# [제공 코드] 4번에서 쓸 프롬프트 — 이 셀은 실행만 하세요.
from langchain_core.prompts import ChatPromptTemplate

wide_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '너는 온라인 서점 고객센터 안내원이다. 아래 자료에 있는 내용만으로 한국어로 간결하게 답한다.\n'
     '자료에 없는 내용은 지어내지 말고 자료에서 찾지 못했다고 답한다.\n\n'
     '자료:\n{context}'),
    ('human', '{question}'),
])
print('프롬프트 준비 완료:', wide_prompt.input_variables)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서를 '하나씩' 잘라야 그 문서에서 몇 번째 조각인지 셀 수 있다(한꺼번에 자르면 문서 경계가 사라진다)
splitter4 = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10,
                                           add_start_index=True)

chunks4 = []
for doc in faq_docs:
    parts = splitter4.split_documents([doc])
    for i, part in enumerate(parts):
        part.metadata['chunk_no'] = i               # 이 문서에서 몇 번째 조각인가
        part.metadata['chunk_total'] = len(parts)   # 이 문서는 모두 몇 조각인가
        chunks4.append(part)

ids4 = [f"{c.metadata['id']}-{c.metadata['chunk_no']}" for c in chunks4]
store4 = Chroma.from_documents(chunks4, embeddings,
                               collection_name='bookstore_chunk_lv2', ids=ids4)
retriever4 = store4.as_retriever(search_kwargs={'k': 1})
print('문서', len(faq_docs), '개 ->', len(chunks4), '조각')


def neighbors4(hit, window=1):
    """검색된 조각의 앞뒤 window 개를 같은 문서에서 꺼내 조각 번호 순서로 돌려준다."""
    m = hit.metadata
    want = [n for n in range(m['chunk_no'] - window, m['chunk_no'] + window + 1)
            if 0 <= n < m['chunk_total']]        # 문서 밖 번호는 거른다
    got = store4.get(where={'$and': [{'id': {'$eq': m['id']}},
                                     {'chunk_no': {'$in': want}}]})
    # get 은 순서를 보장하지 않는다 - 조각 번호로 정렬해야 원문 순서가 된다
    return sorted(zip(got['metadatas'], got['documents']), key=lambda x: x[0]['chunk_no'])


def search_wide4(question):
    """조각 색인에서 검색해, 집힌 조각의 앞뒤까지 이어 붙인 자료를 만든다."""
    hits = retriever4.invoke(question)
    blocks = []
    for h in hits:
        around = neighbors4(h)
        blocks.append(f"[{h.metadata['title']}] " + ' '.join(t for _, t in around))
    return {'context': '\n\n'.join(blocks), 'question': question, 'sources': hits}


def answer_wide4(filled):
    """앞 단계가 만든 재료로 답을 만들고, 답이 본 근거를 그대로 함께 돌려준다."""
    answer = (wide_prompt | model | StrOutputParser()).invoke(filled)
    return {'answer': answer, 'sources': filled['sources']}


# 함수가 파이프를 시작하므로 RunnableLambda 로 직접 감싼다
wide_rag4 = RunnableLambda(search_wide4) | RunnableLambda(answer_wide4)

# wide4 는 '자료가 어떻게 넓어졌는지' 를 눈으로 보려고 앞 단계만 따로 부른 것이다
#   (실제 서비스에서는 체인만 부른다 - 여기서만 확인용으로 한 번 더 부른다)
wide4 = search_wide4('매장 수령은 준비된 뒤 며칠 안에 가야 하나요?')
out4 = wide_rag4.invoke('매장 수령은 준비된 뒤 며칠 안에 가야 하나요?')
print('집힌 조각:', wide4['sources'][0].page_content)   # 여기서 문장이 끊긴다
print('넓힌 자료:', wide4['context'])
print('최종 답  :', out4['answer'])

In [ ]:
# [자가채점]
# (1) 조각 색인 — 문서보다 조각이 많고, 꼬리표에 번호가 달려 있어야 한다
assert len(chunks4) > len(faq_docs), 'chunk_size=50 으로 자르면 조각이 문서보다 많아집니다'
assert all({'chunk_no', 'chunk_total'} <= set(c.metadata) for c in chunks4), \
    '조각마다 chunk_no·chunk_total 을 metadata 에 넣으세요'
assert all(0 <= c.metadata['chunk_no'] < c.metadata['chunk_total'] for c in chunks4)
# 번호는 '문서마다' 0부터 — 한 문서의 번호를 모으면 0,1,2,... 여야 한다
nos_b16 = sorted(c.metadata['chunk_no'] for c in chunks4 if c.metadata['id'] == 'b16')
assert nos_b16 == list(range(len(nos_b16))), 'chunk_no 는 문서마다 0부터 매깁니다'
assert 'b16-1' in store4.get()['ids'], "조각 id 는 '문서id-조각번호' 형태로 만들어 ids 로 넘기세요"

# 이웃 함수 — 같은 문서의 조각만, 번호 순서대로
hit4 = retriever4.invoke('매장 수령은 준비된 뒤 며칠 안에 가야 하나요?')[0]
around4 = list(neighbors4(hit4))
assert len(around4) >= 2, '앞뒤 조각이 함께 나와야 합니다(집힌 조각 하나만 왔습니다)'
assert all(isinstance(x, (tuple, list)) and len(x) == 2 for x in around4), \
    '(metadata, 본문) 쌍의 목록을 돌려주세요'
assert all(m['id'] == hit4.metadata['id'] for m, _ in around4), '같은 문서의 조각만 모으세요'
got_nos4 = [m['chunk_no'] for m, _ in around4]
assert got_nos4 == sorted(got_nos4), 'chunk_no 로 정렬해야 원문 순서가 됩니다'
assert all(0 <= n < hit4.metadata['chunk_total'] for n in got_nos4), '문서 밖 번호는 빼세요'

# (2) 넓은 자료 — 자료는 코드로만 만들어져 결정적이다
assert set(wide4) == {'context', 'question', 'sources'}, \
    "돌려줄 열쇠는 context·question·sources 세 개입니다"
assert wide4['question'] == '매장 수령은 준비된 뒤 며칠 안에 가야 하나요?', '지문의 질문을 그대로 넣어 실행하세요'
assert '7일' not in hit4.page_content, \
    'chunk_size=50, chunk_overlap=10 으로 잘랐는지 확인하세요(집힌 조각은 끊겨 있어야 합니다)'
assert '7일' in wide4['context'], \
    '앞뒤 조각을 이어 붙이지 않았습니다 — 자료가 집힌 조각 하나에 머물러 있습니다'

# 체인 — 모델 문장은 타입·구조로만 본다
assert isinstance(out4, dict) and set(out4) == {'answer', 'sources'}, \
    "체인이 돌려줄 열쇠는 answer 와 sources 입니다"
assert isinstance(out4['answer'], str) and out4['answer'].strip()
assert all(isinstance(d, Document) for d in out4['sources']), 'sources 는 Document 목록입니다'
print('✅ 통과!')

**해설**: 교안 1절과 2절이 여기서 하나로 모입니다.

**왜 조각을 자르면 문맥이 끊기나**: 짧은 조각일수록 질문과 겹치는 말의 비중이 높아 **검색은 더 잘 맞습니다.** 그런데 그렇게 집어 온 조각 하나는 문장 중간에서 잘려 **답을 만들 재료로는 모자랍니다.** 이 문제의 질문이 그 예입니다 — 집힌 조각은 '준비 후' 에서 끝나고, 정작 며칠인지는 **다음 조각**에 있습니다.

**꼬리표가 해법인 이유**: `metadata` 는 검색에 쓰이지 않습니다. **나중에 우리가 쓰려고** 달아 두는 것이고 `chunk_no`·`chunk_total` 이 바로 그 용도입니다 — 앞뒤 번호를 계산하고(`chunk_no`), 문서 밖으로 나가지 않게 막습니다(`chunk_total`). 꺼낼 때는 **검색이 아니라 `get`** 입니다: "뜻이 가까운 글"이 아니라 "같은 문서의 몇 번 조각"이라는 **조건**으로 고르기 때문입니다. 그리고 `get` 은 순서를 보장하지 않으므로 **정렬이 빠지면** 문장이 뒤섞인 채 모델에게 갑니다 — 에러는 나지 않고 **답만 이상해집니다.**

**왜 갈래가 아니라 단계인가**: 1번처럼 `RunnableParallel` 로 답 갈래와 근거 갈래를 나누면 **검색이 두 번** 돕니다. 여기서는 `search_wide4` 가 검색과 이웃 복원을 **한 번에** 끝내고 그 재료를 `answer_wide4` 에 물려줍니다 — 검색은 한 번이고, `sources` 는 답이 **실제로 읽은** 자료입니다. 자료를 더 넓히고 싶으면 `window` 만 키우면 되고 **답을 만드는 단계는 그대로**입니다.

**출력에서 같은 말이 두 번 보이는 이유**: `chunk_overlap=10` 때문입니다 — 조각 경계의 몇 글자가 앞뒤 조각에 모두 남아 있어, 이어 붙이면 그 자리에서 되풀이됩니다(`'매장 수령은 매장 수령은'`). 사람 눈에는 거슬려도 모델이 뜻을 파악하는 데는 문제가 없습니다.

**공짜는 아닙니다**: 자료가 넓어지면 프롬프트가 길어지고 토큰이 늘며, 관계없는 문장도 함께 들어옵니다. `k` 를 고르던 것과 **같은 저울질**입니다.

**채점 기준**: 모델 문장은 실행마다 다르므로 **타입·구조**만 봅니다. 반면 `wide4['context']` 는 **코드로만 만들어져 결정적**이라, 집힌 조각에 없던 '7일' 이 자료에 들어왔는지까지 확인합니다 — 이웃을 실제로 붙였다는 증거입니다.

---
수고했어요! RAG 를 **체인으로 직접 조립**해 근거까지 함께 받았습니다. 데이터베이스를 도구로 붙여 **자연어 질문을 SQL 로** 바꾸고 그 답까지 **스키마에 담아** 받았으며, 외부 API 를 도구로 감싸며 **실패를 문자열로 돌려주는** 습관도 손에 익혔습니다. 마지막으로 조각으로 잘린 문서에서 **앞뒤 문맥을 되살려 답을 완성**했습니다. LV3 에서는 이것들을 묶어 **리뷰 인텔리전스 파이프라인**을 만듭니다.